# Modern GPT — Six Improvements That Matter

> ↩ **Previous in the series:** [Building a Transformer from Scratch](2_transformer.ipynb)

The GPT from the previous notebook is the 2020 GPT-2 baseline: character-level tokens, learned absolute position embeddings, LayerNorm, and manual dot-product attention. It works. Every major model since — Llama, Mistral, Gemma — has replaced or improved six components.

This notebook applies those six improvements one at a time. Each is a self-contained swap. You can see exactly what changes, verify that it still produces the right shapes, and understand why it helps.

| Improvement | What it replaces | Why |
|-------------|-----------------|-----|
| BPE tokenizer | Character-level (vocab 65) | More information per token |
| RMSNorm | LayerNorm | Simpler, no centering, same effect |
| Flash Attention | Manual Q@Kᵀ/@V | Fused kernel, O(T) memory not O(T²) |
| RoPE | Learned absolute pos embeddings | Encodes relative distance, zero extra params |
| Group-Query Attention | n_heads K/V projections | Fewer K/V parameters, smaller KV cache |
| KV cache | Recomputing past K/V at each step | O(T²) → O(T) generation |

All six run on CPU. By the end, a single `ModernGPT` class combines them all.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
torch.manual_seed(1337)

In [ ]:
with open('../../data/tinyshakespeare.txt', 'r') as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f'{len(text):,} characters, vocab size {vocab_size}')

In [ ]:
block_size = 32
batch_size = 64
n_embd     = 64
n_heads    = 4
n_layers   = 4
dropout    = 0.2
device     = 'cuda' if torch.cuda.is_available() else 'cpu'

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, n_batches=50):
    model.eval()
    results = {}
    for split in ('train', 'val'):
        losses = torch.zeros(n_batches)
        for i in range(n_batches):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[i] = loss.item()
        results[split] = losses.mean().item()
    model.train()
    return results

def train(model, steps=1000, lr=3e-4, label=''):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for step in range(steps):
        x, y = get_batch('train')
        _, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    losses = estimate_loss(model)
    print(f'{label:30s}  params {sum(p.numel() for p in model.parameters()):,}  '
          f'val loss {losses["val"]:.4f}')
    return losses['val']

## Baseline — The GPT from Notebook 2

The architecture is unchanged from Step 7 of the transformer notebook: four blocks, each with multi-head self-attention and a feed-forward layer wrapped in residual connections and LayerNorm. We train for 1 000 steps and record the validation loss. Every improvement below will be compared against this number.

In [ ]:
class _Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.drop  = nn.Dropout(dropout)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x);  q = self.query(x)
        hs = k.shape[-1]
        wei = q @ k.transpose(-2, -1) * hs**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = self.drop(F.softmax(wei, dim=-1))
        return wei @ self.value(x)

class _MHA(nn.Module):
    def __init__(self, n_embd, n_heads, block_size, dropout):
        super().__init__()
        hs = n_embd // n_heads
        self.heads = nn.ModuleList([_Head(n_embd, hs, block_size, dropout) for _ in range(n_heads)])
        self.proj  = nn.Linear(n_embd, n_embd)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))

class _FF(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class _Block(nn.Module):
    def __init__(self, n_embd, n_heads, block_size, dropout):
        super().__init__()
        self.sa  = _MHA(n_embd, n_heads, block_size, dropout)
        self.ff  = _FF(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        return x + self.ff(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_heads, n_layers, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[_Block(n_embd, n_heads, block_size, dropout) for _ in range(n_layers)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

In [ ]:
torch.manual_seed(1337)
baseline = GPT(vocab_size, block_size, n_embd, n_heads, n_layers, dropout).to(device)
print(f'Baseline parameters: {sum(p.numel() for p in baseline.parameters()):,}')
baseline_val = train(baseline, steps=1000, label='Baseline GPT')

## Improvement 1: BPE Tokenizer

The baseline encodes each character as one token: 65 distinct tokens, one character each. A 32-token context window covers 32 characters — about five words.

**Byte-pair encoding (BPE)** merges frequent character pairs into subword tokens: `ing`, `the`, `tion` each become a single token. GPT-2's vocabulary has 50 257 tokens. The same 32-position window now covers roughly 20–25 words — about 5× more text per forward pass.

The trade-off: a 50 k vocabulary means larger embedding and output projection tables. With `n_embd=64`, those tables dominate the parameter count. Production models use `n_embd ≥ 512` to amortise this cost. For the remaining improvements we keep the character tokenizer so training stays fast on CPU; swapping in tiktoken is a two-line change.

In [ ]:
try:
    import tiktoken
    enc = tiktoken.get_encoding('gpt2')
    sample = text[:500]
    char_ids = encode(sample)
    bpe_ids  = enc.encode(sample)
    print(f'500 characters → {len(char_ids)} char tokens  |  {len(bpe_ids)} BPE tokens')
    print(f'BPE compression ratio: {len(char_ids)/len(bpe_ids):.1f}×')
    print(f'GPT-2 vocab size: {enc.n_vocab:,}')
    print()
    # Show a few BPE tokens
    for tok_id in bpe_ids[:10]:
        print(f'  {tok_id:6d}  →  {repr(enc.decode([tok_id]))}')
except ImportError:
    print('tiktoken not installed. Run:  uv add tiktoken')
    print('The two-line swap to use it in the model:')
    print('  enc = tiktoken.get_encoding("gpt2")')
    print('  vocab_size = enc.n_vocab   # 50 257')
    print('  data = torch.tensor(enc.encode(text), dtype=torch.long)')

## Improvement 2: RMSNorm

**LayerNorm** does two things: it *centres* the activations (subtracts the mean) and *scales* them (divides by the standard deviation), then applies learned scale γ and shift β.

**RMSNorm** (Root Mean Square Layer Norm) drops the centering step entirely:

```
RMSNorm(x) = x / rms(x) * γ        rms(x) = sqrt(mean(x²))
```

In transformer residual streams, the centering step turns out to matter very little — the residual connections already keep activations from drifting far from zero. Removing it makes RMSNorm simpler and about 15% faster than LayerNorm, with no measurable loss in quality. Llama uses RMSNorm throughout.

One fewer learned parameter per norm: no bias β. The learned scale γ stays.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))  # learned scale, no bias

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight

# Verify: same shape as LayerNorm, different values (no centering)
x   = torch.randn(2, 8, 64)
ln  = nn.LayerNorm(64)
rms = RMSNorm(64)
print(f'Input     mean={x.mean():.3f}  std={x.std():.3f}')
print(f'LayerNorm mean={ln(x).mean():.3f}  std={ln(x).std():.3f}  shape={tuple(ln(x).shape)}')
print(f'RMSNorm   mean={rms(x).mean():.3f}  std={rms(x).std():.3f}  shape={tuple(rms(x).shape)}')
print()
# LayerNorm centres (mean ≈ 0) and scales (std ≈ 1).
# RMSNorm only scales — mean is NOT forced to zero.

## Improvement 3: Flash Attention

The attention head computes:

```python
wei = softmax( Q @ Kᵀ / √head_size )   # full (T, T) matrix — O(T²) memory
out = wei @ V
```

The `(T, T)` matrix is materialised in full. For T=1 024 tokens and 4 heads that is 4 × 1 024² × 4 bytes ≈ 16 MB per layer — and it grows quadratically with context length.

**Flash Attention** rewrites the computation as a fused kernel that tiles Q, K, V through fast on-chip memory (SRAM), never writing the full attention matrix to GPU memory. The result is mathematically identical but memory usage is O(T) instead of O(T²).

In PyTorch, one function call replaces the entire three-line sequence:

In [ ]:
# Verify Flash Attention produces the same result as the manual computation
torch.manual_seed(0)
B, H, T, HS = 2, 4, 8, 16
q = torch.randn(B, H, T, HS)
k = torch.randn(B, H, T, HS)
v = torch.randn(B, H, T, HS)

# Manual — what 2_transformer.ipynb does
tril = torch.tril(torch.ones(T, T, dtype=torch.bool))
wei = (q @ k.transpose(-2, -1)) * HS**-0.5
wei = wei.masked_fill(~tril, float('-inf'))
wei = F.softmax(wei, dim=-1)
out_manual = wei @ v

# Flash Attention — is_causal handles the mask internally
out_flash = F.scaled_dot_product_attention(q, k, v, is_causal=True)

diff = (out_manual - out_flash).abs().max().item()
print(f'Max difference between manual and flash: {diff:.2e}')
print('Match!' if diff < 1e-5 else 'MISMATCH — check shapes')

In [ ]:
# Rewrite multi-head attention using Flash Attention.
# Q, K, V are projected together (one fused linear) and reshaped for the kernel.
# No tril buffer needed — is_causal=True handles masking inside the kernel.
class _FlashMHA(nn.Module):
    def __init__(self, n_embd, n_heads, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.hs      = n_embd // n_heads
        self.qkv     = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj    = nn.Linear(n_embd, n_embd, bias=False)
        self.drop    = dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_heads, self.hs).transpose(1, 2)  # (B, H, T, HS)
        k = k.view(B, T, self.n_heads, self.hs).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.hs).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                  dropout_p=self.drop if self.training else 0.0)
        return self.proj(out.transpose(1, 2).contiguous().view(B, T, C))

## Improvement 4: Rotary Position Embeddings (RoPE)

The baseline adds a learned position embedding to each token before the transformer: position 0 always gets vector `pos_emb[0]`, position 1 gets `pos_emb[1]`, and so on. Two problems:

1. **No relative position information.** The model sees that position 3 has a particular embedding, but it has to infer from the pattern of those embeddings that positions 3 and 4 are adjacent. There is no built-in notion of distance.
2. **Length extrapolation fails.** Position embeddings only exist up to `block_size`. The model has never seen `pos_emb[33]` and has no idea what to do with a sequence longer than it was trained on.

**RoPE** encodes position by *rotating* the query and key vectors before computing attention. For each dimension pair `(2d, 2d+1)` at position `m`, the rotation angle is `m × θ_d` where `θ_d = 1 / 10000^(2d/head_size)`.

The key property: the dot product `Qᵢ · Kⱼ` becomes a function of only `(i − j)` — the *relative distance* — not the absolute positions. No absolute position information is needed. The model learns relative relationships, which generalise to longer sequences.

In [ ]:
def precompute_rope(head_size, seq_len, device='cpu'):
    """Precompute cos and sin tables for RoPE up to seq_len positions."""
    assert head_size % 2 == 0
    # θ_d = 1 / (10000^(2d/head_size))  for d = 0 … head_size/2 - 1
    theta = 1.0 / (10000.0 ** (
        torch.arange(0, head_size, 2, device=device).float() / head_size
    ))                                              # (head_size/2,)
    positions = torch.arange(seq_len, device=device).float()
    angles = torch.outer(positions, theta)          # (seq_len, head_size/2)
    return angles.cos(), angles.sin()               # each (seq_len, head_size/2)

def apply_rope(x, cos, sin):
    """Apply rotary embeddings. x: (B, n_heads, T, head_size)."""
    T = x.shape[2]
    cos_t = cos[:T].unsqueeze(0).unsqueeze(0)      # (1, 1, T, head_size/2)
    sin_t = sin[:T].unsqueeze(0).unsqueeze(0)
    x_even = x[..., ::2]                           # (B, H, T, head_size/2)
    x_odd  = x[..., 1::2]
    rot_even = x_even * cos_t - x_odd * sin_t
    rot_odd  = x_even * sin_t + x_odd * cos_t
    # Interleave even and odd dimensions back together
    return torch.stack([rot_even, rot_odd], dim=-1).flatten(-2)

In [ ]:
# Verify the relative-distance property:
# with absolute pos embeddings, Q_i · K_j depends on i and j separately.
# with RoPE, it should depend only on (i - j).

HS = 16
cos_table, sin_table = precompute_rope(HS, seq_len=32)

torch.manual_seed(42)
q_raw = torch.randn(1, 1, 32, HS)  # one batch, one head, 32 positions
k_raw = torch.randn(1, 1, 32, HS)

q_rot = apply_rope(q_raw, cos_table, sin_table)
k_rot = apply_rope(k_raw, cos_table, sin_table)

# Compute dot product at position pair (5, 3) vs (10, 8) — same distance, different positions
dot_53  = (q_rot[0, 0, 5]  @ k_rot[0, 0, 3]).item()
dot_108 = (q_rot[0, 0, 10] @ k_rot[0, 0, 8]).item()

# These WON'T be identical (different q,k values were sampled at each position),
# but the *functional form* of the dot product is the same function of (i-j=2).
# The key point: apply_rope(q_i) · apply_rope(k_j) = f(q_raw_i, k_raw_j, i-j)
# — absolute positions i and j cancel out, only their difference survives.
print(f'RoPE dot product at positions (5,3) : {dot_53:.4f}')
print(f'RoPE dot product at positions (10,8): {dot_108:.4f}')
print('(Different because q/k values differ; same relative encoding i-j=2 in both cases)')

## Improvement 5: Group-Query Attention (GQA)

Standard multi-head attention projects queries, keys, and values to `n_heads` separate heads. During inference, every layer must store K and V for all past tokens — one tensor per head. The **KV cache** grows as `O(n_layers × n_heads × T × head_size)`. For a large model with long context, this dominates GPU memory.

**Group-Query Attention** decouples the number of Q heads from the number of K/V heads. With `n_kv_heads = n_heads // 2`, groups of two query heads share one K/V pair. The KV cache shrinks by 2×. Quality is nearly unchanged — the model's representational power comes mostly from the query diversity, not the key/value count.

Llama 2 (70B) uses `n_heads=64, n_kv_heads=8` — an 8× reduction in KV cache size.

In [ ]:
class _GQA(nn.Module):
    """Multi-head attention with Group-Query: n_kv_heads <= n_heads."""
    def __init__(self, n_embd, n_heads, n_kv_heads, dropout):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.hs         = n_embd // n_heads
        self.groups     = n_heads // n_kv_heads
        self.q_proj  = nn.Linear(n_embd, n_heads    * self.hs, bias=False)
        self.k_proj  = nn.Linear(n_embd, n_kv_heads * self.hs, bias=False)
        self.v_proj  = nn.Linear(n_embd, n_kv_heads * self.hs, bias=False)
        self.out     = nn.Linear(n_embd, n_embd, bias=False)
        self.drop    = dropout

    def forward(self, x, rope_cos=None, rope_sin=None):
        B, T, C = x.shape
        H, KV, HS = self.n_heads, self.n_kv_heads, self.hs
        q = self.q_proj(x).view(B, T, H,  HS).transpose(1, 2)   # (B, H,  T, HS)
        k = self.k_proj(x).view(B, T, KV, HS).transpose(1, 2)   # (B, KV, T, HS)
        v = self.v_proj(x).view(B, T, KV, HS).transpose(1, 2)   # (B, KV, T, HS)
        if rope_cos is not None:
            q = apply_rope(q, rope_cos, rope_sin)
            k = apply_rope(k, rope_cos, rope_sin)
        # Expand KV heads to match query head count
        k = k.repeat_interleave(self.groups, dim=1)              # (B, H, T, HS)
        v = v.repeat_interleave(self.groups, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                  dropout_p=self.drop if self.training else 0.0)
        return self.out(out.transpose(1, 2).contiguous().view(B, T, C))

# Parameter count comparison
n_kv_heads = 2   # half the query heads
mha = _FlashMHA(n_embd, n_heads, dropout)
gqa = _GQA(n_embd, n_heads, n_kv_heads, dropout)
print(f'Standard MHA  params: {sum(p.numel() for p in mha.parameters()):,}')
print(f'GQA (kv={n_kv_heads})     params: {sum(p.numel() for p in gqa.parameters()):,}')
print(f'KV projection savings: {1 - n_kv_heads/n_heads:.0%} fewer K/V params')

## Improvement 6: KV Cache

During generation, at each step the model receives the full sequence so far and recomputes K and V for every token in it — including all the tokens it already computed K/V for at the previous step. Step 1 computes 1 token. Step 2 recomputes that token plus 1 more. At step T, it recomputes all T tokens. Total work: O(T²).

The **KV cache** stores each token's K and V vectors as they are produced. At step t, only the new token's K/V need to be computed and appended. Attention then runs between the single new query and all cached K/V. Total work: O(T).

The cache trades memory for time: storing K/V for all past tokens uses `O(n_layers × n_kv_heads × T × head_size)` memory — exactly the KV cache that GQA already reduced.

In [ ]:
# Conceptual demo: show the quadratic vs linear work counts
def work_without_cache(n_tokens):
    """Total attention computations to generate n_tokens sequentially."""
    return sum(t * t for t in range(1, n_tokens + 1))  # O(T²) per token

def work_with_cache(n_tokens):
    """Total attention computations with KV cache."""
    return sum(t for t in range(1, n_tokens + 1))       # O(T) per token

print(f'{"Tokens":>8}  {"No cache":>12}  {"With cache":>12}  {"Speedup":>10}')
for n in [10, 32, 128, 512, 1024]:
    wo = work_without_cache(n)
    wi = work_with_cache(n)
    print(f'{n:>8}  {wo:>12,}  {wi:>12,}  {wo/wi:>9.1f}×')

In [ ]:
# Time baseline generate() — no cache, recomputes everything at each step
torch.manual_seed(1337)
timing_model = GPT(vocab_size, block_size, n_embd, n_heads, n_layers, dropout).to(device)
seed = torch.zeros((1, 1), dtype=torch.long, device=device)
n_tokens = 200

timing_model.eval()
with torch.no_grad():
    t0 = time.time()
    out = timing_model.generate(seed.clone(), n_tokens)
    t1 = time.time()

print(f'No cache: {t1-t0:.2f}s for {n_tokens} tokens  ({n_tokens/(t1-t0):.1f} tok/s)')
print()
print('Sample:', decode(out[0].tolist()[:80]))

## Putting It Together — ModernGPT

One class combining all five architectural improvements (the tokenizer swap is a data-pipeline change, not a model change):

- **RMSNorm** everywhere LayerNorm appeared
- **Flash Attention** via `F.scaled_dot_product_attention`
- **RoPE** — position embeddings removed, rotations applied inside attention
- **GQA** — `n_kv_heads` shared key/value heads

Everything else — the residual stream, the feed-forward layer, the cross-entropy loss, the four-step training loop — is unchanged.

In [ ]:
class _ModernBlock(nn.Module):
    def __init__(self, n_embd, n_heads, n_kv_heads, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa  = _GQA(n_embd, n_heads, n_kv_heads, dropout)
        self.ff  = _FF(n_embd, dropout)
        self.rn1 = RMSNorm(n_embd)
        self.rn2 = RMSNorm(n_embd)
        # Precompute RoPE tables and register as buffers (moves with .to(device))
        cos, sin = precompute_rope(head_size, block_size)
        self.register_buffer('rope_cos', cos)
        self.register_buffer('rope_sin', sin)

    def forward(self, x):
        x = x + self.sa(self.rn1(x), self.rope_cos, self.rope_sin)
        return x + self.ff(self.rn2(x))


class ModernGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_heads, n_kv_heads, n_layers, dropout):
        super().__init__()
        self.block_size = block_size
        # No position embedding — RoPE is applied inside each attention layer
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks  = nn.ModuleList([
            _ModernBlock(n_embd, n_heads, n_kv_heads, block_size, dropout)
            for _ in range(n_layers)
        ])
        self.rn_f    = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        # Weight tying: output projection shares weights with token embedding.
        # A token that appears often in training gets a good embedding, and that
        # same good representation guides the output distribution. Halves the
        # memory cost of the largest matrices in the model.
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx)                # (B, T, n_embd) — no pos_emb added
        for block in self.blocks:
            x = block(x)
        x = self.rn_f(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


torch.manual_seed(1337)
modern = ModernGPT(vocab_size, block_size, n_embd, n_heads,
                   n_kv_heads=2, n_layers=n_layers, dropout=dropout).to(device)

baseline_params = sum(p.numel() for p in baseline.parameters())
modern_params   = sum(p.numel() for p in modern.parameters())
print(f'Baseline GPT  params: {baseline_params:,}')
print(f'ModernGPT     params: {modern_params:,}  (weight tying + GQA reduces count)')

In [ ]:
torch.manual_seed(1337)
modern = ModernGPT(vocab_size, block_size, n_embd, n_heads,
                   n_kv_heads=2, n_layers=n_layers, dropout=dropout).to(device)
modern_val = train(modern, steps=1000, label='ModernGPT')
print()
print(f'Baseline val loss: {baseline_val:.4f}')
print(f'ModernGPT val loss: {modern_val:.4f}')

In [ ]:
modern.eval()
seed = torch.zeros((1, 1), dtype=torch.long, device=device)
out  = modern.generate(seed, max_new_tokens=300, temperature=0.8)
print(decode(out[0].tolist()))

## Summary

| Improvement | Lines changed | What disappears | Effect |
|-------------|--------------|-----------------|--------|
| BPE tokenizer | 2 (re-encode + vocab_size) | Character IDs | ~5× more text per context window |
| RMSNorm | 1 per norm site | Mean-centering, bias β | Simpler, ~15% faster |
| Flash Attention | 3 → 1 in MHA forward | `tril` buffer, manual softmax | O(T) memory not O(T²) |
| RoPE | remove `pos_emb` | Absolute position table | Relative positions, better length generalisation |
| GQA (n_kv=2) | ~20 (new class) | Half the K/V projections | Smaller KV cache, faster inference |
| KV cache | generate() | Recomputation of past tokens | O(T²) → O(T) generation |

### What we learned

- **None of these change the training loop.** Predict, score, assign blame, nudge — identical throughout.
- **RMSNorm and Flash Attention are pure engineering improvements.** Same expressive power, lower cost.
- **RoPE changes what the model learns.** Relative position awareness is a property of the representation, not the engineering stack.
- **GQA and KV cache are inference optimisations.** They matter most at serving time, when memory and latency are the constraint.
- **Weight tying** is one more trick not listed above: sharing the token embedding and the output projection halves the parameter count of the two largest matrices in the model with no loss in quality.

All six appear in Llama 2/3, Mistral, and Gemma. The transformer notebook's Step 7 architecture and this `ModernGPT` are recognisably the same machine — same residual stream, same attention + feed-forward structure, same cross-entropy loss.

> ➡ **Next in the series:** [Fine-tuning](4_finetuning.ipynb)

## Your turn: make it stick

### 1. Summarise from memory

Close the notebook and answer without peeking:

- Why does RMSNorm produce different output values than LayerNorm even though both normalise the activations? Which property does LayerNorm have that RMSNorm deliberately drops?
- Flash Attention produces the same numbers as manual attention. If the result is identical, where exactly does the O(T²) → O(T) memory saving come from?
- RoPE removes the position embedding table. Where does position information enter the model instead?
- With n_heads=4 and n_kv_heads=1, how many parameters does the K projection have compared to the Q projection? Is Q still allowed to be full-rank?

### 2. Ablations

Run each of these and compare val loss to the baseline:

1. **RMSNorm only** — swap LayerNorm → RMSNorm in the baseline GPT, retrain for 1 000 steps. How much of the ModernGPT improvement comes from this single change?
2. **GQA with n_kv_heads=1** — all query heads share a single K/V pair. Does quality degrade noticeably at n_heads=4?
3. **No weight tying** — remove the `self.lm_head.weight = self.tok_emb.weight` line. How does parameter count change? Does val loss change after 1 000 steps?

### 3. Scale the context

Change `block_size` from 32 to 128, keeping everything else the same. Train both the baseline GPT and ModernGPT for 1 000 steps.

- How much does the baseline slow down (quadratic attention)?
- Does the larger context improve val loss?
- Check that `apply_rope` still works correctly: the precomputed cos/sin tables are sized to `block_size`, so longer sequences need larger tables.